In [0]:
print("Test")

In [0]:
from pathlib import Path
import numpy as np  
import pandas as pd
import pickle  
import re      


#Create a data folder for the files generated by this notebook.
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

print(f"Data folder is ready: {DATA_DIR.resolve()}")

## First attempt to get information from one URL

In [0]:
import requests
from bs4 import BeautifulSoup
import json
import re

url = "https://www.rottentomatoes.com/m/i_love_boosters"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36"
}

resp = requests.get(url, headers=headers, timeout=10)

if resp.status_code != 200:
    print("Error HTTP:", resp.status_code)
else:
    soup = BeautifulSoup(resp.content, "html.parser")

    title = None
    h1 = soup.find("h1")
    if h1:
        title = h1.get_text(strip=True)
    else:
        og = soup.find("meta", property="og:title")
        if og and og.get("content"):
            title = og["content"]

    print(f"Título: {title}")

    tomatometer = None
    popcornmeter = None

    score_board = soup.find("score-board") or soup.find("scoreboard") or soup.find("scoreBoard")

    if score_board:
        tomatometer = (
            score_board.get("tomatometerscore")
            or score_board.get("tomatometer")
        )

        popcornmeter = (
            score_board.get("popcornmeterscore")
            or score_board.get("popcornmeter")
            or score_board.get("audiencescore")
            or score_board.get("audienceScore")
        )

    # Fallback por texto visible: "63% Tomatometer ... 89% Popcornmeter"
    page_text = soup.get_text(" ", strip=True)

    if tomatometer is None:
        match = re.search(r"(\d+)%\s+Tomatometer", page_text, re.I)
        if match:
            tomatometer = match.group(1) + "%"

    if popcornmeter is None:
        match = re.search(r"(\d+)%\s+Popcornmeter", page_text, re.I)
        if match:
            popcornmeter = match.group(1) + "%"

    print(f"Tomatometer: {tomatometer}")
    print(f"Popcornmeter: {popcornmeter}")

## Second attempt to get complete information from one URL

In [0]:
import requests
from bs4 import BeautifulSoup
import json
from urllib.parse import urljoin
import pandas as pd

url = "https://www.rottentomatoes.com/m/star_wars_the_mandalorian_and_grogu"

# url = "https://www.rottentomatoes.com/m/mile_end_kicks"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36"
}
resp = requests.get(url, headers=headers, timeout=10)
if resp.status_code != 200:
    print("Error HTTP:", resp.status_code)
else:
    soup = BeautifulSoup(resp.content, "html.parser")


###################### Título (fallbacks) ################################
    title = None
    h1 = soup.find("h1")
    if h1:
        title = h1.get_text(strip=True)
    else:
        og = soup.find("meta", property="og:title")
        if og and og.get("content"):
            title = og["content"]
    print(f"Título: {title}")  


############# Determinar si es película o TV show ######################
if "rottentomatoes.com/m/" in url:
    content_type = "Movie"
elif "rottentomatoes.com/tv/" in url:
    content_type = "TV Show"
else:
    content_type = "Unknown"



######################## Critics And summary ########################

script = soup.find("script", id="media-scorecard-json")

if script and script.string:
    data = json.loads(script.string)

    summary = data.get("description")

    tomatometer = data.get("criticsScore", {}).get("scorePercent")
    popcornmeter = data.get("audienceScore", {}).get("scorePercent")

    print("Resumen:", summary)
    print("Tomatometer:", tomatometer)
    print("Popcornmeter:", popcornmeter)


################## Platform Names ######################################
page_text = soup.get_text(" ", strip=True)

platform_names = []

for li in soup.find_all("li", attrs={"data-qa": "movies-at-home-item"}):
    a = li.find("a", href=True)

    if a and "affiliates:" in a["href"]:
        name = a.get_text(strip=True)
        platform_names.append(name)

platform_names = list(dict.fromkeys(platform_names))

print(platform_names)


####################### Critics consensus #############################

critics_consensus = None

# Opción 1: buscar por id
consensus_div = soup.find("div", id="critics-consensus")

if consensus_div:
    p = consensus_div.find("p")
    if p:
        critics_consensus = p.get_text(" ", strip=True)



print("Critics Consensus:", critics_consensus, '\n')

################# Dictionary of each movie to create my Dataset
rows = []

movie_row = {
    "title": title,
    "content_type": content_type,
    "url": url, 
    "tomatometer": tomatometer,
    "popcornmeter": popcornmeter,
    "summary": summary,
    "critics_consensus": critics_consensus,
    "platforms": ", ".join(platform_names)
}

#########Creating the dataframe
print(movie_row)
rows.append(movie_row)
df = pd.DataFrame(rows)
df


In [0]:
## Check the values per page to check how many pages will be scrolled
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36"
}

all_movies_urls = []

for page in range(1,10):  # 1 hasta 10
    url = f"https://www.rottentomatoes.com/browse/tv_series_browse/?page={page}"
    # print(f"Scrapeando página {page}: {url}")

    resp = requests.get(url, headers=headers, timeout=10)

    if resp.status_code != 200:
        print(f"Error en página {page}: HTTP {resp.status_code}")
        continue

    soup = BeautifulSoup(resp.content, "html.parser")

    page_movies = []

    for a in soup.find_all("a", href=True):
        full_url = urljoin(url, a["href"])

        if "rottentomatoes.com/tv/" in full_url:
            page_movies.append(full_url)

    page_movies = list(dict.fromkeys(page_movies))
    print(f"Series encontradas en página {page}: {len(page_movies)}")

    all_movies_urls.extend(page_movies)

    time.sleep(1)

all_movies_urls = list(dict.fromkeys(all_movies_urls))

##### Summarize of all movies founded
print(f"\nTotal películas únicas: {len(all_movies_urls)}")
for movie in all_movies_urls:
    print(movie)
# At the end there are Just 154 unic values, so it could be scrolled until page 5

In [0]:
# Path for files pkl wich are smaller than JSON files
# This File will contain the dataset of Series
pickle_path = DATA_DIR / "series_urls.pkl"

In [0]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36"
}

all_series_urls = []

for page in range(1, 5):  # 1 hasta 11
    url = f"https://www.rottentomatoes.com/browse/tv_series_browse/?page={page}"
    # print(f"Scrapeando página {page}: {url}")

    resp = requests.get(url, headers=headers, timeout=10)

    if resp.status_code != 200:
        print(f"Error en página {page}: HTTP {resp.status_code}")
        continue

    soup = BeautifulSoup(resp.content, "html.parser")

    for a in soup.find_all("a", href=True):
        full_url = urljoin(url, a["href"])

        if "rottentomatoes.com/tv/" in full_url:
            all_series_urls.append(full_url)

    time.sleep(1)  # pausa pequeña para no saturar la página

# Quitar duplicados
all_series_urls = list(dict.fromkeys(all_series_urls))

print(f"\nTotal de series encontradas: {len(all_series_urls)}")

for u in all_series_urls:
    print(u)


with open(pickle_path, "wb") as f:
    pickle.dump(all_series_urls, f)



In [0]:
### Just for verify the data
# df_imported = pd.read_pickle(pickle_path)
#df_imported

#### USER-AGENT REQUEST -----> TO TAKE ALL THE DATA FROM THE URLs

In [0]:
import requests
from bs4 import BeautifulSoup
import json
from urllib.parse import urljoin
import pandas as pd
import time


# "User-Agent" REQUEST simula que la petición como si viniera desde un navegador Chrome en Windows para evitar bloqueos de HTML
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36"
}



###########################    DEFINICIÓN DE MI FUNCIÓN PARA CADA URL    ###########################
def scrape_movie(url):
    resp = requests.get(url, headers=headers, timeout=10) ## Uso de el agente para hacer la solicitud como si viniera del navegador

    if resp.status_code != 200:
        print(f"Error HTTP {resp.status_code}: {url}")
        return None

    soup = BeautifulSoup(resp.content, "html.parser")

    # Título 
    title = None
    h1 = soup.find("h1")

    if h1:
        title = h1.get_text(strip=True)
    else:
        og = soup.find("meta", property="og:title")
        if og and og.get("content"):
            title = og["content"]

####################### 0. DETERMINE CONTENT TYPE (Movie or TV Show)

    if "rottentomatoes.com/m/" in url:
        content_type = "Movie"
    elif "rottentomatoes.com/tv/" in url:
        content_type = "TV Show"
    else:
        content_type = "Unknown"

####################### 1 . CALLING GENRE(S) 

    genre = None # create the variable

    script = soup.find("script", id="mps-page-integration")

    if script:
        script_text = script.get_text()

        match = re.search(r'"cag\[genre\]":"([^"]+)"', script_text) #Looking for the text

        if match:
            genre = match.group(1)
    
    genres = [] #Save genres

    if genre:
        genres = genre.split("|") # Spplit them 

    # print("Genre:", genre)
    
################# 2. CALLING: Summary, Tomatometer, Popcornmeter

     # Variables 
    summary = None
    tomatometer = None
    popcornmeter = None

    script = soup.find("script", id="media-scorecard-json")

    if script:
        try:
            data = json.loads(script.get_text(strip=True))
            
            #Checking for texts, different forms to get the values
            summary = data.get("description")
            tomatometer = data.get("criticsScore", {}).get("scorePercent")
            popcornmeter = data.get("audienceScore", {}).get("scorePercent")

        except json.JSONDecodeError:
            pass

 ############### 3. STREAM PLATFORMS 
    platform_names = []

    for li in soup.find_all("li", attrs={"data-qa": "movies-at-home-item"}):
        a = li.find("a", href=True)

        if a and "affiliates:" in a["href"]:
            name = a.get_text(strip=True)
            platform_names.append(name)

    platform_names = list(dict.fromkeys(platform_names))

############### 4. CRITICS CONSENSUS
    critics_consensus = None

    consensus_div = soup.find("div", id="critics-consensus")

    if consensus_div:
        p = consensus_div.find("p")
        if p:
            critics_consensus = p.get_text(" ", strip=True)

    movie_row = {
        "title": title,
        "content_type": content_type,
        "genre": genre,
        "url": url,
        "tomatometer": tomatometer,
        "popcornmeter": popcornmeter,
        "summary": summary,
        "critics_consensus": critics_consensus,
        "platforms": ", ".join(platform_names)
    }

    return movie_row ## Me retorna el valor de cada columna nueva


################################# Fin de la función ######################

In [0]:
#Cargo mi archivo pkl ### MAS ADELANTE SE ACTUALIZA CON TODAS LAS SERIES Y PEL[ICULAS]
with pickle_path.open("rb") as f:
    all_series_urls = pickle.load(f)

In [0]:
#####  For para crear la matriz
rows = []

for i, url in enumerate(all_series_urls, start=1):
    # print(f"Scrapeando {i}/{len(movie_urls)}: {url}")

    movie_row = scrape_movie(url)

    if movie_row is not None:
        rows.append(movie_row)

    time.sleep(1)


#### Creación del DataFrame 
df = pd.DataFrame(rows)



#### Guardar EL dataframe
df.to_csv("rottentomatoes_series_dataset.csv", index=False, encoding="utf-8")

df.head()

In [0]:
## Check the values per page to check how many pages will be scrolled
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36"
}

all_movies_urls = []

for page in range(1,3):  # 1 hasta 10
    url = f"https://www.rottentomatoes.com/browse/movies_in_theaters/sort:popular?page={page}"
    # print(f"Scrapeando página {page}: {url}")

    resp = requests.get(url, headers=headers, timeout=10)

    if resp.status_code != 200:
        print(f"Error en página {page}: HTTP {resp.status_code}")
        continue

    soup = BeautifulSoup(resp.content, "html.parser")

    page_movies = []

    for a in soup.find_all("a", href=True):
        full_url = urljoin(url, a["href"])

        if "rottentomatoes.com/m/" in full_url:
            page_movies.append(full_url)

    page_movies = list(dict.fromkeys(page_movies))
    print(f"Series encontradas en página {page}: {len(page_movies)}")

    all_movies_urls.extend(page_movies)

    time.sleep(1)

all_movies_urls = list(dict.fromkeys(all_movies_urls))

##### Summarize of all movies founded
print(f"\nTotal películas únicas: {len(all_movies_urls)}")
for movie in all_movies_urls:
    print(movie)
# At the end there are Just 54 unic values, so it could be scrolled until page 3

# SCRAPEANDO POR PÁGINAS EN ROTTENTOMATOS -----  Movies In teathers

In [0]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import json
import time

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36"
}

all_movies_urls = []

for page in range(1, 3):  # 1 hasta 11
    url = f"https://www.rottentomatoes.com/browse/movies_in_theaters/sort:popular?page={page}"
    # print(f"Scrapeando página {page}: {url}")

    resp = requests.get(url, headers=headers, timeout=10)

    if resp.status_code != 200:
        print(f"Error en página {page}: HTTP {resp.status_code}")
        continue

    soup = BeautifulSoup(resp.content, "html.parser")

    for a in soup.find_all("a", href=True):
        full_url = urljoin(url, a["href"])

        if "rottentomatoes.com/m/" in full_url:
            all_movies_urls.append(full_url)

    time.sleep(1)  # pausa pequeña para no saturar la página

# Quitar duplicados
all_movies_urls = list(dict.fromkeys(all_movies_urls))

print(f"\nTotal de peliculas encontradas: {len(all_movies_urls)}")

####Printing all URLs
# for u in all_movies_urls:
#     print(u)

###Creation of the Json document
# with open("movies_urls_pages_1_to_11.json", "w", encoding="utf-8") as f:
#     json.dump(all_movies_urls, f, indent=4, ensure_ascii=False)

pickle_path = DATA_DIR / "movies_urls_teathre.pkl"

with open(pickle_path, "wb") as f:
    pickle.dump(all_movies_urls, f)

### Creando el Dataset final

In [0]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import json
import time

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36"
}

all_movies_urls = []

for page in range(1, 2):  # 1 hasta 11
    url = f"https://www.rottentomatoes.com/browse/movies_at_home/={page}"
    #print(f"Scrapeando página {page}: {url}")

    resp = requests.get(url, headers=headers, timeout=10)

    if resp.status_code != 200:
        print(f"Error en página {page}: HTTP {resp.status_code}")
        continue

    soup = BeautifulSoup(resp.content, "html.parser")

    for a in soup.find_all("a", href=True):
        full_url = urljoin(url, a["href"])

        if "rottentomatoes.com/m/" in full_url:
            all_movies_urls.append(full_url)

    time.sleep(1)  # pausa pequeña para no saturar la página

# Quitar duplicados
all_movies_urls = list(dict.fromkeys(all_movies_urls))

print(f"\nTotal de peliculas encontradas: {len(all_movies_urls)}")

# for u in all_movies_urls:
#     print(u)

pickle_path = DATA_DIR / "movies_urls_home.pkl"

with open(pickle_path, "wb") as f:
    pickle.dump(all_movies_urls, f)


In [0]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import json
import time

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36"
}

all_movies_urls = []

for page in range(1, 6):  # 1 hasta 11
    url = f"https://www.rottentomatoes.com/browse/movies_coming_soon/?page={page}"
    # print(f"Scrapeando página {page}: {url}")

    resp = requests.get(url, headers=headers, timeout=10)

    if resp.status_code != 200:
        print(f"Error en página {page}: HTTP {resp.status_code}")
        continue

    soup = BeautifulSoup(resp.content, "html.parser")

    for a in soup.find_all("a", href=True):
        full_url = urljoin(url, a["href"])

        if "rottentomatoes.com/m/" in full_url:
            all_movies_urls.append(full_url)

    time.sleep(1)  # pausa pequeña para no saturar la página

# Quitar duplicados
all_movies_urls = list(dict.fromkeys(all_movies_urls))

print(f"\nTotal de peliculas encontradas: {len(all_movies_urls)}")

# Esto solo muestra/imprime cada URL en pantalla, dentro del notebook. No guarda nada en un archivo
# for u in all_movies_urls:
#     print(u)

pickle_path = DATA_DIR / "movies_urls_soon.pkl"

with open(pickle_path, "wb") as f:
    pickle.dump(all_movies_urls, f)


#### USER-AGENT REQUEST -----> TO Create the Dataset with all the URLs

In [0]:
import requests
from bs4 import BeautifulSoup
import json
from urllib.parse import urljoin
import pandas as pd
import time


# "User-Agent" REQUEST simula que la petición como si viniera desde un navegador Chrome en Windows para evitar bloqueos de HTML
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36"
}



###########################    DEFINICIÓN DE MI FUNCIÓN PARA CADA URL    ###########################
def scrape_movie(url):
    resp = requests.get(url, headers=headers, timeout=10) ## Uso de el agente para hacer la solicitud como si viniera del navegador

    if resp.status_code != 200:
        print(f"Error HTTP {resp.status_code}: {url}")
        return None

    soup = BeautifulSoup(resp.content, "html.parser")

    # Título
    title = None
    h1 = soup.find("h1")

    if h1:
        title = h1.get_text(strip=True)
    else:
        og = soup.find("meta", property="og:title")
        if og and og.get("content"):
            title = og["content"]

####################### 0. DETERMINE CONTENT TYPE (Movie or TV Show)

    if "rottentomatoes.com/m/" in url:
        content_type = "Movie"
    elif "rottentomatoes.com/tv/" in url:
        content_type = "TV Show"
    else:
        content_type = "Unknown"

####################### 1 . CALLING GENRE(S) 

    genre = None # create the variable

    script = soup.find("script", id="mps-page-integration")

    if script:
        script_text = script.get_text()

        match = re.search(r'"cag\[genre\]":"([^"]+)"', script_text) #Looking for the text

        if match:
            genre = match.group(1)
    
    genres = [] #Save genres

    if genre:
        genres = genre.split("|") # Spplit them 

    # print("Genre:", genre)
    
################# 2. CALLING: Summary, Tomatometer, Popcornmeter

     # Variables 
    summary = None
    tomatometer = None
    popcornmeter = None

    script = soup.find("script", id="media-scorecard-json")

    if script:
        try:
            data = json.loads(script.get_text(strip=True))
            
            #Checking for texts, different forms to get the values
            summary = data.get("description")
            tomatometer = data.get("criticsScore", {}).get("scorePercent")
            popcornmeter = data.get("audienceScore", {}).get("scorePercent")

        except json.JSONDecodeError:
            pass

 ############### 3. STREAM PLATFORMS 
    platform_names = []

    for li in soup.find_all("li", attrs={"data-qa": "movies-at-home-item"}):
        a = li.find("a", href=True)

        if a and "affiliates:" in a["href"]:
            name = a.get_text(strip=True)
            platform_names.append(name)

    platform_names = list(dict.fromkeys(platform_names))

############### 4. CRITICS CONSENSUS
    critics_consensus = None

    consensus_div = soup.find("div", id="critics-consensus")

    if consensus_div:
        p = consensus_div.find("p")
        if p:
            critics_consensus = p.get_text(" ", strip=True)

    movie_row = {
        "title": title,
        "content_type": content_type,
        "genre": genre,
        "url": url,
        "tomatometer": tomatometer,
        "popcornmeter": popcornmeter,
        "summary": summary,
        "critics_consensus": critics_consensus,
        "platforms": ", ".join(platform_names)
    }

    return movie_row ## Me retorna el valor de cada columna nueva


################################# Fin de la función ######################

In [0]:
# #Cargo mi archivo pkl ### MAS ADELANTE SE ACTUALIZA CON TODAS LAS SERIES Y PEL[ICULAS]
# with pickle_path.open("rb") as f:
#     all_series_urls = pickle.load(f)


pickle_path1 = DATA_DIR / "movies_urls_teathre.pkl"
pickle_path2 = DATA_DIR / "series_urls.pkl"
pickle_path3 = DATA_DIR / "movies_urls_home.pkl"
pickle_path4 = DATA_DIR / "movies_urls_soon.pkl"

with open(pickle_path1, "rb") as f:
    list1 = pickle.load(f)
with open(pickle_path2, "rb") as f:
    list2 = pickle.load(f)
with open(pickle_path3, "rb") as f:
    list3 = pickle.load(f)
with open(pickle_path4, "rb") as f:
    list4 = pickle.load(f)    

combined_list = list1 + list2 + list3 + list4
combined_list = list(dict.fromkeys(combined_list))  # Remover duplicados
# combined_list

In [0]:

#####  For para crear la matriz
rows = []

for i, url in enumerate(combined_list, start=1):
    # print(f"Scrapeando {i}/{len(movie_urls)}: {url}")

    movie_row = scrape_movie(url)

    if movie_row is not None:
        rows.append(movie_row)

    time.sleep(1)


#### Creación del DataFrame 
df = pd.DataFrame(rows)



#### Guardar EL dataframe
df.to_csv("rottentomatoes_series_dataset.csv", index=False, encoding="utf-8")

df.head()


# Whole URL scrap at the same time

In [0]:
from pathlib import Path
import numpy as np  
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import json
import pickle
import time

#Create a data folder for the files generated by this notebook.
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

print(f"Data folder is ready: {DATA_DIR.resolve()}")

pickle_path = DATA_DIR / "all_pages.pkl"


search_bases = [
    "https://www.rottentomatoes.com/browse/movies_at_home/affiliates:fandango-at-home",
    "https://www.rottentomatoes.com/browse/movies_at_home/affiliates:netflix",
    "https://www.rottentomatoes.com/browse/movies_at_home/affiliates:apple-tv-plus",
    "https://www.rottentomatoes.com/browse/movies_at_home/affiliates:prime-video",
    "https://www.rottentomatoes.com/browse/movies_at_home/sort:popular",
    "https://www.rottentomatoes.com/browse/movies_at_home/critics:certified_fresh",
    "https://www.rottentomatoes.com/browse/movies_at_home/",
    "https://www.rottentomatoes.com/browse/movies_in_theaters/sort:newest",
    "https://www.rottentomatoes.com/browse/movies_at_home/sort:popular",
    "https://www.rottentomatoes.com/browse/tv_series_browse/sort:popular",
    
]

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36"
}

all_series_urls = []
count_error = 0

for base_url in search_bases:
    for page in range(1, 12):  # páginas 1 a 11
        url = f"{base_url}?page={page}"
        # print(f"Scrapeando página {page}: {url}")

        resp = requests.get(url, headers=headers, timeout=10)

        
        if resp.status_code != 200:
            count_error += 1
            ##### Error if the page has not been created
            # print(f"Error en página {page} de {base_url}: HTTP {resp.status_code}") 
            #print(count_error)
            continue # If the error happens, continue with next page

        soup = BeautifulSoup(resp.content, "html.parser")

        for a in soup.find_all("a", href=True):
            full_url = urljoin(url, a["href"])

            if "rottentomatoes.com/tv/" in full_url:
                all_series_urls.append(full_url)
            elif"rottentomatoes.com/m/" in full_url:
                all_series_urls.append(full_url)

        time.sleep(1)  # pausa pequeña para no saturar la página

# Quitar duplicados
all_series_urls = list(dict.fromkeys(all_series_urls))


print(f"\nTotal de series encontradas: {len(all_series_urls)}")
# for u in all_series_urls:
#     print(u)

with open(DATA_DIR / "all_pages.pkl", "wb") as f:
    pickle.dump(all_series_urls, f)


In [0]:
import pandas as pd
import pickle

# Cargar el archivo pickle
with open(DATA_DIR / "all_pages.pkl", "rb") as f:
    urls_list = pickle.load(f)

# Convertir a DataFrame
df = pd.DataFrame(urls_list, columns=["url"])
print(df)
print(df.shape)

In [0]:
df.to_csv(DATA_DIR / "all_pages.csv", index=False)

In [0]:
import requests
from bs4 import BeautifulSoup
import json
from urllib.parse import urljoin
import pandas as pd
import time
import re


# "User-Agent" REQUEST simula que la petición como si viniera desde un navegador Chrome en Windows para evitar bloqueos de HTML
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0 Safari/537.36"
}



###########################    DEFINICIÓN DE MI FUNCIÓN PARA CADA URL    ###########################
def scrape_movie(url):
    resp = requests.get(url, headers=headers, timeout=10) ## Uso de el agente para hacer la solicitud como si viniera del navegador

    if resp.status_code != 200:
        print(f"Error HTTP {resp.status_code}: {url}")
        return None

    soup = BeautifulSoup(resp.content, "html.parser")

    # Título
    title = None
    h1 = soup.find("h1")

    if h1:
        title = h1.get_text(strip=True)
    else:
        og = soup.find("meta", property="og:title")
        if og and og.get("content"):
            title = og["content"]

####################### 0. DETERMINE CONTENT TYPE (Movie or TV Show)

    if "rottentomatoes.com/m/" in url:
        content_type = "Movie"
    elif "rottentomatoes.com/tv/" in url:
        content_type = "TV Show"
    else:
        content_type = "Unknown"

####################### 1 . CALLING GENRE(S) 

    genre = None # create the variable

    script = soup.find("script", id="mps-page-integration")

    if script:
        script_text = script.get_text()

        match = re.search(r'"cag\[genre\]":"([^"]+)"', script_text) #Looking for the text

        if match:
            genre = match.group(1)
    
    genres = [] #Save genres

    if genre:
        genres = genre.split("|") # Spplit them 

    # print("Genre:", genre)
    
################# 2. CALLING: Summary, Tomatometer, Popcornmeter

     # Variables 
    summary = None
    tomatometer = None
    popcornmeter = None

    script = soup.find("script", id="media-scorecard-json")

    if script:
        try:
            data = json.loads(script.get_text(strip=True))
            
            #Checking for texts, different forms to get the values
            summary = data.get("description")
            tomatometer = data.get("criticsScore", {}).get("scorePercent")
            popcornmeter = data.get("audienceScore", {}).get("scorePercent")

        except json.JSONDecodeError:
            pass

 ############### 3. STREAM PLATFORMS 
    platform_names = []

    for li in soup.find_all("li", attrs={"data-qa": "movies-at-home-item"}):
        a = li.find("a", href=True)

        if a and "affiliates:" in a["href"]:
            name = a.get_text(strip=True)
            platform_names.append(name)

    platform_names = list(dict.fromkeys(platform_names))

############### 4. CRITICS CONSENSUS
    critics_consensus = None

    consensus_div = soup.find("div", id="critics-consensus")

    if consensus_div:
        p = consensus_div.find("p")
        if p:
            critics_consensus = p.get_text(" ", strip=True)

    movie_row = {
        "title": title,
        "content_type": content_type,
        "genre": genre,
        "url": url,
        "tomatometer": tomatometer,
        "popcornmeter": popcornmeter,
        "summary": summary,
        "critics_consensus": critics_consensus,
        "platforms": ", ".join(platform_names)
    }

    return movie_row ## Me retorna el valor de cada columna nueva


################################# Fin de la función ######################

In [0]:
# #Cargo mi archivo pkl ### MAS ADELANTE SE ACTUALIZA CON TODAS LAS SERIES Y PEL[ICULAS]
# with pickle_path.open("rb") as f:
#     all_series_urls = pickle.load(f)


pickle_path1 = DATA_DIR / "all_pages.pkl"


with open(pickle_path1, "rb") as f:
    list1 = pickle.load(f)


combined_list = list1
combined_list = list(dict.fromkeys(combined_list))  # Remover duplicados
# combined_list

In [0]:

#####  For para crear la matriz
rows = []

for i, url in enumerate(combined_list, start=1):
    # print(f"Scrapeando {i}/{len(movie_urls)}: {url}")

    movie_row = scrape_movie(url)

    if movie_row is not None:
        rows.append(movie_row)

    time.sleep(1)


#### Creación del DataFrame 
df = pd.DataFrame(rows)



#### Guardar EL dataframe
df.to_csv("rottentomatoes_series_dataset.csv", index=False, encoding="utf-8")

df.head()


## Call and transform in dataset

In [0]:
from pathlib import Path
import pandas as pd
import json
import pickle
import numpy as np  



#Create a data folder for the files generated by this notebook.
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

print(f"Data folder is ready: {DATA_DIR.resolve()}")


synthetic_csv_path = DATA_DIR / "all_pages.csv"
synthetic_json_path = DATA_DIR / "all_pages.json"
pickle_path = DATA_DIR / "all_pages.pkl"

## List of URLs
df_imported = pd.read_pickle(pickle_path)
type(df_imported)
print(df_imported)

# df_imported.to_csv(synthetic_csv_path, index=False)
# df_imported.to_json(synthetic_json_path, orient="records", indent=4)

# Transdformed as  Dataframe
# df_imported = pd.DataFrame(df_imported)
# print(df_imported.shape)
# print(df_imported)
# type(df_imported)

In [0]:
from pathlib import Path
import pandas as pd
import json
import pickle
import numpy as np  



#Create a data folder for the files generated by this notebook.
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

print(f"Data folder is ready: {DATA_DIR.resolve()}")


synthetic_csv_path = DATA_DIR / "all_pages.csv"
# synthetic_json_path = DATA_DIR / "all_pages.json"
# pickle_path = DATA_DIR / "all_pages.pkl"

## List of URLs
df_imported = pd.read_csv("rottentomatoes_series_dataset.csv")
type(df_imported)
print(df_imported)
